# Vesuvius Surface Detection -- 1000-epoch scroll-grouped LOSO submission

Runs inference with a single trained nnU-Net checkpoint and produces `submission.zip`.

**Real submission, matched ablation against the earlier 100-epoch baseline** (public
0.46426 / private 0.46559, submission `55294106`). Same LOSO split (scroll 26010 held out
entirely, 657 train / 129 val), same seed (42), same everything except training length --
1000 epochs instead of 100, isolating epoch count as the only variable changed. See
`baselinerun/research_log.md` for the full local-vs-real calibration story from the 100-epoch
run (local held-out 0.5162 vs. real Kaggle private 0.4656) and the reasoning for why this run
is a fresh 1000-epoch run rather than a continuation of the 100-epoch checkpoint (nnU-Net's
LR schedule decays to ~0 by the declared epoch count; continuing into a different target
would reintroduce that decayed LR at a much higher value -- a real discontinuity, not
equivalent to a fresh run at the new length).

Adapted from the training pipeline in `baselinerun/` (a local reproduction of
`surface-nnunet-training-inference-with-2xt4.ipynb`) and structured following the working
offline-install + GPU-inference pattern demonstrated in
`nnunet-4-model-7-5-2-1-final-submit-so-long.ipynb`. Every function below is a verbatim copy
of the corresponding `baselinerun/src/...` module -- see that repo for the tested,
documented originals; this notebook only exists because Kaggle submissions must run through a
Notebook with internet disabled, so the code has to be self-contained here rather than
imported as a package.

**Before running on Kaggle**, attach as notebook inputs:
- The competition dataset (`vesuvius-challenge-surface-detection`) -- provided automatically.
- `CHECKPOINT_DATASET_SLUG` -- your uploaded checkpoint dataset (see `baselinerun/kaggle_submission/checkpoint_1000epoch_v1/`).
- `WHEELS_DATASET_SLUG` -- your uploaded offline wheel bundle (see `baselinerun/kaggle_submission/wheels_v3/`).

## Configuration

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import zipfile
from pathlib import Path
from typing import Optional, Tuple, Union

# ---- Kaggle dataset slugs -- UPDATE THESE to match what you actually upload ----
CHECKPOINT_DATASET_SLUG = "vesuvius-1000epoch-checkpoint-v1"   # kaggle datasets create result
WHEELS_DATASET_SLUG = "vesuvius-nnunet-wheels-v3"  # kaggle datasets create result

# ---- Paths (Kaggle mounts) ----
# NOTE: competition/other-users' data mounts under /kaggle/input/competitions/<slug> and
# /kaggle/input/datasets/<owner>/<slug> respectively -- NOT the flat /kaggle/input/<slug> path
# every "mount timeout" earlier was chasing. Confirmed empirically via a CPU diagnostic kernel
# that walked /kaggle/input directly. Our OWN datasets (same account as the kernel) happen to
# also be reachable at the flat path, which is why those always "just worked".
INPUT_DIR = Path("/kaggle/input/competitions/vesuvius-challenge-surface-detection")
CHECKPOINT_DIR = Path(f"/kaggle/input/{CHECKPOINT_DATASET_SLUG}")
WHEELS_DIR = Path(f"/kaggle/input/{WHEELS_DATASET_SLUG}")
WORKING_DIR = Path("/kaggle/temp")
OUTPUT_DIR = Path("/kaggle/working")

NNUNET_RAW = WORKING_DIR / "nnUNet_data" / "nnUNet_raw"
NNUNET_PREPROCESSED = WORKING_DIR / "nnUNet_data" / "nnUNet_preprocessed"
NNUNET_RESULTS = WORKING_DIR / "nnUNet_results"

# ---- Model identity (must match what was trained -- see baselinerun/configs/training_baseline.yaml) ----
DATASET_ID = 100
DATASET_NAME = "Dataset100_VesuviusSurface"
CONFIGURATION = "3d_lowres"
PLANS_NAME = "nnUNetResEncUNetMPlans"
TRAINER = "nnUNetTrainerSeeded"  # custom trainer, flexible-epochs variant -- see EXT_TRAINER_DIR cell below
CHECKPOINT_FILENAME = "checkpoint_best.pth"  # EMA-best by pseudo-dice, not the last epoch
FOLD = 0  # scroll-grouped split: fold 0 = scroll 26010 held out (see splits_final.json)

MODEL_DIR_NAME = f"{TRAINER}__{PLANS_NAME}__{CONFIGURATION}"

# nnUNetTrainerSeeded isn't a stock nnU-Net class -- nnUNetv2_predict still needs to
# resolve it by name (the checkpoint embeds its own trainer_name, and inference re-derives the
# network architecture via that class) even though the seeding logic in __init__ never runs at
# inference time. Written to disk and pointed at via nnUNet_extTrainer in the cell below.
EXT_TRAINER_DIR = WORKING_DIR / "ext_trainers"

TEST_INPUT_DIR = WORKING_DIR / "test_input"
PREDICTIONS_DIR = WORKING_DIR / "predictions"
PREDICTIONS_TIFF_DIR = OUTPUT_DIR / "predictions_tiff"
SUBMISSION_ZIP = OUTPUT_DIR / "submission.zip"

print("INPUT_DIR:", INPUT_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("WHEELS_DIR:", WHEELS_DIR)


## Offline install

Real internet is disabled during Kaggle submission scoring. Installs `nnunetv2==2.8.1`
(exactly the version the checkpoint was trained with -- avoids any train/inference version
drift) plus its non-torch dependencies from the offline wheel bundle. torch/torchvision are
deliberately *not* bundled: Kaggle's GPU notebook image already ships a working torch, and
pulling a mismatched one from PyPI's default index during wheel-building actually resolved to
the wrong CUDA toolkit version entirely -- see `baselinerun/kaggle_submission/README.md`.

The wheel bundle is built for **Python 3.12** (`cp312`) -- Kaggle's actual kernel Python,
confirmed from a real failed run's traceback paths (`/usr/local/lib/python3.12/...`), not
assumed. The first build targeted 3.11 to match the local dev env and failed on Kaggle with
`ERROR: Could not find a version that satisfies the requirement nnunetv2==2.8.1 (from
versions: none)` -- pip correctly refused to install `cp311`-tagged compiled wheels
(numpy/scipy/scikit-image/etc.) into a 3.12 interpreter.

In [ ]:
import time


def wait_for_mount(path: Path, min_files: int = 1, timeout_s: int = 900, poll_s: int = 15) -> None:
    """Dataset attachments can lag behind a `kaggle datasets create/version` push finishing --
    the upload API call returns before Kaggle's backend has finished processing the dataset,
    so a kernel that starts immediately after can see /kaggle/input/<slug> as not-yet-mounted.
    Hit this twice in a row (both attempts failed identically on WHEELS_DIR specifically, well
    after the dataset was confirmed to have real content via the Kaggle API) -- this is a wait
    for Kaggle's own mounting, not a bug in the dataset itself."""
    waited = 0
    while waited <= timeout_s:
        if path.exists() and len(list(path.iterdir())) >= min_files:
            print(f"{path} mounted ({len(list(path.iterdir()))} entries) after {waited}s")
            return
        time.sleep(poll_s)
        waited += poll_s
    raise RuntimeError(f"{path} not mounted after {timeout_s}s (exists={path.exists()})")


wait_for_mount(WHEELS_DIR, min_files=50)
wait_for_mount(CHECKPOINT_DIR, min_files=1)

result = subprocess.run(
    f"pip install --no-index --find-links={WHEELS_DIR} nnunetv2==2.8.1 nibabel tifffile tqdm -q",
    shell=True, capture_output=True, text=True,
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-3000:])
    raise RuntimeError("Offline install failed")

import nnunetv2
print("nnunetv2 installed OK")


## Environment setup

Verbatim from `baselinerun/src/training/environment.py::setup_environment`.

In [ ]:
def setup_environment():
    for d in [NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS, OUTPUT_DIR]:
        d.mkdir(parents=True, exist_ok=True)

    os.environ["nnUNet_raw"] = str(NNUNET_RAW)
    os.environ["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED)
    os.environ["nnUNet_results"] = str(NNUNET_RESULTS)
    # baselinerun's own config uses "true" (torch.compile), but Kaggle's GPU pool can assign
    # older cards (confirmed: Tesla P100, CUDA capability 6.0) that torch.compile's Triton
    # backend does not support (requires >=7.0) -- a real run hit
    # "torch._inductor.exc.GPUTooOldForTriton" here. Disabled for Kaggle specifically;
    # arunodhayan's own notebook independently made the same call for the same reason.
    os.environ["nnUNet_compile"] = "false"

    print(f"nnUNet_raw: {NNUNET_RAW}")
    print(f"nnUNet_preprocessed: {NNUNET_PREPROCESSED}")
    print(f"nnUNet_results: {NNUNET_RESULTS}")
    print(f"nnUNet_USE_BLOSC2: {os.environ.get('nnUNet_USE_BLOSC2', 'not set')} (0=NPZ, 1=blosc2)")


setup_environment()


## Stage the checkpoint and the custom trainer

Copies the uploaded checkpoint dataset into the layout `nnUNetv2_predict` expects:
`nnUNet_results/{DATASET_NAME}/{Trainer}__{Plans}__{Config}/{dataset.json,plans.json,fold_0/checkpoint_best.pth}`.

The checkpoint dataset is uploaded *flat* (dataset.json, plans.json, checkpoint_best.pth all
at the dataset root, no subfolder) deliberately -- an earlier version uploaded a nested folder
via `kaggle datasets create -r zip`, which bundles the whole folder as a single `.zip` file
rather than unpacking it, and that dataset consistently failed to mount into a running kernel
at all (confirmed: still not mounted after a 15-minute wait, while flat, non-zipped datasets
mounted instantly). Flat upload avoids the zip path entirely.

Also writes `nnUNetTrainerSeeded.py` to disk and sets `nnUNet_extTrainer` so
`nnUNetv2_predict` can resolve `nnUNetTrainerSeeded` by name (the checkpoint embeds
this trainer name internally; inference re-derives the network architecture via
`recursive_find_trainer_class_by_name`, exactly the same external-trainer mechanism used
during training -- see `baselinerun/src/training/nnunet_trainers/nnUNetTrainerSeeded.py`).

In [ ]:
dst_model_dir = NNUNET_RESULTS / DATASET_NAME / MODEL_DIR_NAME
dst_model_dir.mkdir(parents=True, exist_ok=True)
(dst_model_dir / f"fold_{FOLD}").mkdir(parents=True, exist_ok=True)

shutil.copy2(CHECKPOINT_DIR / "dataset.json", dst_model_dir / "dataset.json")
shutil.copy2(CHECKPOINT_DIR / "plans.json", dst_model_dir / "plans.json")
shutil.copy2(CHECKPOINT_DIR / CHECKPOINT_FILENAME, dst_model_dir / f"fold_{FOLD}" / CHECKPOINT_FILENAME)

checkpoint_path = dst_model_dir / f"fold_{FOLD}" / CHECKPOINT_FILENAME
assert checkpoint_path.exists(), f"Missing checkpoint at {checkpoint_path}"
print("Staged checkpoint:", checkpoint_path, f"({checkpoint_path.stat().st_size / 1e6:.1f} MB)")
print("dataset.json:", (dst_model_dir / "dataset.json").exists())
print("plans.json:", (dst_model_dir / "plans.json").exists())

# ---- Stage the custom trainer (verbatim from baselinerun's own file) so nnUNetv2_predict can
# resolve nnUNetTrainerSeeded by name via the external-trainer mechanism. ----
EXT_TRAINER_DIR.mkdir(parents=True, exist_ok=True)
(EXT_TRAINER_DIR / "nnUNetTrainerSeeded.py").write_text('''\
"""Seeded nnU-Net trainer variant, for reproducible baseline runs.

Stock nnU-Net sets no random seed anywhere. Loaded via nnU-Net's external-trainer mechanism
(env var nnUNet_extTrainer) rather than by editing the installed nnunetv2 package -- see
nnunetv2.utilities.find_objects.recursive_find_trainer_class_by_name. At inference time only
the class needs to resolve and build_network_architecture (inherited, unchanged) needs to
work -- __init__ (and its seeding) never runs, since nnUNetv2_predict calls
build_network_architecture on the class directly without instantiating it.
"""

from __future__ import annotations

import os
import random

import numpy as np
import torch
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer


def _seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class nnUNetTrainerSeeded(nnUNetTrainer):
    """nnUNetTrainer (1000-epoch stock default) + a fixed RNG seed (read from env var NNUNET_SEED, default 42)."""

    def __init__(self, plans: dict, configuration: str, fold: int, dataset_json: dict,
                 device: torch.device = torch.device("cuda")):
        self.seed = int(os.environ.get("NNUNET_SEED", 42))
        _seed_everything(self.seed)
        print(f"[nnUNetTrainerSeeded] seeded torch/numpy/random with seed={self.seed}")
        super().__init__(plans, configuration, fold, dataset_json, device)
''')
os.environ["nnUNet_extTrainer"] = str(EXT_TRAINER_DIR)
print("Staged custom trainer at:", EXT_TRAINER_DIR / "nnUNetTrainerSeeded.py")
print("nnUNet_extTrainer:", os.environ["nnUNet_extTrainer"])


## Prepare test data

Verbatim from `baselinerun/src/data/prepare_training_data.py`
(`create_spacing_json`, `prepare_single_case`, `prepare_test_data`) -- only the test-data
subset of that module is needed here, since this notebook only runs inference.

In [ ]:
from tqdm.auto import tqdm


def create_spacing_json(output_path: Path, shape: tuple, spacing: tuple = (1.0, 1.0, 1.0)):
    json_data = {"spacing": list(spacing)}
    with open(output_path, "w") as f:
        json.dump(json_data, f)


def prepare_single_case(src_path: Path, dest_path: Path, json_path: Path, use_symlinks: bool = True) -> bool:
    try:
        import tifffile
        with tifffile.TiffFile(src_path) as tif:
            shape = tif.pages[0].shape if len(tif.pages) == 1 else (len(tif.pages), *tif.pages[0].shape)

        if use_symlinks:
            if not dest_path.exists():
                dest_path.symlink_to(src_path.resolve())
        else:
            shutil.copy2(src_path, dest_path)

        create_spacing_json(json_path, shape)
        return True
    except Exception as e:
        print(f"Error processing {src_path.name}: {e}")
        return False


def prepare_test_data(input_dir: Path, output_dir: Path, use_symlinks: bool = True) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    test_images_dir = input_dir / "test_images"
    if not test_images_dir.exists():
        raise FileNotFoundError(f"{test_images_dir} not found")

    test_files = sorted(test_images_dir.glob("*.tif"))
    print(f"Found {len(test_files)} test cases")

    for img_path in tqdm(test_files, desc="Preparing test data"):
        case_id = img_path.stem
        prepare_single_case(img_path, output_dir / f"{case_id}_0000.tif", output_dir / f"{case_id}_0000.json", use_symlinks)

    return output_dir


# The competition dataset (attached via competition_sources, not dataset_sources) previously
# hit the exact same mount-lag issue as our own datasets -- confirmed via the Kaggle API that
# test_images/1407735.tif genuinely exists, yet a run still saw test_images/ never mount even
# after a full 15-minute wait (unlike our own datasets, which always mounted instantly once
# uploaded correctly). Diagnose broadly this time instead of assuming the fix is identical:
# wait for INPUT_DIR itself first, print what's actually there, then handle test_images
# specifically with clear diagnostics either way.
wait_for_mount(INPUT_DIR, min_files=1, timeout_s=300)
print("INPUT_DIR contents:", sorted(p.name for p in INPUT_DIR.iterdir()))

test_images_dir = INPUT_DIR / "test_images"
try:
    wait_for_mount(test_images_dir, min_files=1, timeout_s=600)
except RuntimeError as e:
    print(f"WARNING: {e}")
    print("Trying case-insensitive / alternate-name search under INPUT_DIR...")
    candidates = [p for p in INPUT_DIR.rglob("*") if p.is_dir() and "test" in p.name.lower()]
    print("Directories with 'test' in the name:", candidates)
    for c in candidates:
        try:
            print(f"  {c}: {sorted(p.name for p in c.iterdir())[:10]}")
        except Exception as inner_e:
            print(f"  {c}: could not list ({inner_e})")
    raise

prepare_test_data(INPUT_DIR, TEST_INPUT_DIR)


## Run inference

`_run_command` and `run_inference` verbatim from `baselinerun/src/training/commands.py`.

In [ ]:
def _run_command(cmd: str, name: str = "Command", tail_lines: int = 30, timeout: Optional[int] = None) -> bool:
    print(f"Running: {cmd}")
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        print(f"{name} TIMEOUT after {timeout}s!")
        return False

    if result.returncode != 0:
        print(f"{name} FAILED!")
        print(f"STDERR:\n{result.stderr[-3000:]}")
        return False

    print(f"{name} complete!")
    if result.stdout.strip():
        lines = result.stdout.strip().split("\n")
        print("\n".join(lines[-tail_lines:]))
    return True


def run_inference(
    input_dir: Path, output_dir: Path, dataset_id: int, config: str, fold: Union[int, str],
    plans: str, trainer: str, checkpoint_name: str = "checkpoint_final.pth", save_probabilities: bool = True,
    num_processes_preprocessing: int = 2, num_processes_segmentation: int = 2,
) -> bool:
    output_dir.mkdir(parents=True, exist_ok=True)
    cmd = f"nnUNetv2_predict -d {dataset_id:03d} -c {config} -f {fold}"
    cmd += f" -i {input_dir} -o {output_dir} -p {plans} -tr {trainer} -chk {checkpoint_name}"
    cmd += f" -npp {num_processes_preprocessing} -nps {num_processes_segmentation}"
    cmd += " --verbose"
    if save_probabilities:
        cmd += " --save_probabilities"
    return _run_command(cmd, "Inference")


ok = run_inference(
    input_dir=TEST_INPUT_DIR, output_dir=PREDICTIONS_DIR,
    dataset_id=DATASET_ID, config=CONFIGURATION, fold=FOLD,
    plans=PLANS_NAME, trainer=TRAINER, checkpoint_name=CHECKPOINT_FILENAME,
)
assert ok, "Inference failed -- see STDERR above"


## Convert predictions to submission TIFFs

`load_probabilities` and `predictions_to_tiff` verbatim from
`baselinerun/src/training/postprocess.py` (the `.nii.gz` fallback branch is dropped here --
it exists in the source only for a legacy prediction path this notebook never produces).

In [ ]:
import numpy as np
import tifffile


def load_probabilities(npz_path: Path) -> np.ndarray:
    data = np.load(npz_path)
    return data["probabilities"]


def predictions_to_tiff(pred_dir: Path, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)
    npz_files = list(pred_dir.glob("*.npz"))
    tif_files = list(pred_dir.glob("*.tif"))

    if npz_files:
        print(f"Converting {len(npz_files)} NPZ probability files to TIFF...")
        for npz_path in tqdm(npz_files, desc="Converting to TIFF"):
            case_id = npz_path.stem
            probs = load_probabilities(npz_path)
            pred = np.argmax(probs, axis=0).astype(np.uint8)
            tifffile.imwrite(output_dir / f"{case_id}.tif", pred)
    elif tif_files:
        print(f"Copying {len(tif_files)} TIFF prediction files...")
        for tif_path in tqdm(tif_files, desc="Copying TIFF"):
            case_id = tif_path.stem
            pred = tifffile.imread(str(tif_path)).astype(np.uint8)
            tifffile.imwrite(output_dir / f"{case_id}.tif", pred)
    else:
        print(f"WARNING: No prediction files found in {pred_dir}")


predictions_to_tiff(PREDICTIONS_DIR, PREDICTIONS_TIFF_DIR)


## Sanity check: dimensions and dtype

Not part of the source pipeline -- added here because the competition rules are explicit
that each mask "must match the dimensions of the source image exactly, and use the same data
type as the train mask" (uint8). Cheap to check, expensive to get wrong.

In [ ]:
train_labels_dir = INPUT_DIR / "train_labels"
sample_train_label = next(train_labels_dir.glob("*.tif"))
expected_dtype = tifffile.imread(str(sample_train_label)).dtype
print(f"Expected dtype (from a train label): {expected_dtype}")

all_ok = True
for pred_path in sorted(PREDICTIONS_TIFF_DIR.glob("*.tif")):
    case_id = pred_path.stem
    src_path = INPUT_DIR / "test_images" / f"{case_id}.tif"
    pred_arr = tifffile.imread(str(pred_path))
    src_arr = tifffile.imread(str(src_path))

    shape_ok = pred_arr.shape == src_arr.shape
    dtype_ok = pred_arr.dtype == expected_dtype
    all_ok &= shape_ok and dtype_ok

    print(f"{case_id}: pred shape={pred_arr.shape} dtype={pred_arr.dtype} | "
          f"src shape={src_arr.shape} | shape_ok={shape_ok} dtype_ok={dtype_ok}")

assert all_ok, "Dimension/dtype mismatch detected -- fix before submitting"
print("\nAll predictions match source dimensions and expected dtype.")


## Generate submission.zip

Verbatim from `baselinerun/src/training/submission.py::generate_submission`.

In [ ]:
def generate_submission(predictions_tiff_dir: Path, output_zip: Path, delete_after_zip: bool = True) -> Optional[Path]:
    tiff_files = sorted(predictions_tiff_dir.glob("*.tif"))
    if not tiff_files:
        print(f"No TIFF files found in {predictions_tiff_dir}")
        return None

    print(f"Creating submission ZIP with {len(tiff_files)} files...")
    with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as zipf:
        for tiff_path in tqdm(tiff_files, desc="Zipping predictions"):
            zipf.write(tiff_path, tiff_path.name)
            if delete_after_zip:
                tiff_path.unlink()

    zip_size_mb = output_zip.stat().st_size / (1024 * 1024)
    print(f"Submission saved: {output_zip} ({zip_size_mb:.1f} MB)")
    return output_zip


submission_path = generate_submission(PREDICTIONS_TIFF_DIR, SUBMISSION_ZIP)
assert submission_path is not None and submission_path.exists()
print("\nDone:", submission_path)
